I did Variant effect prediction using https://www.ensembl.org/Tools/VEP

In [1]:
# resulting vcf file - changing format from all in one columns to multiple 

# remove the first 10 rows with descriptions
import pandas as pd
df = pd.read_csv("vr_output.vcf",sep="\t",skiprows=11)
df = df[df['ALT'] != 'CCC']
df.to_csv("vr_output1.vcf",sep='\t')

# i dropped the insertion that is not present in my dataset so i don't confuse the merge

# method to expand columns
def expand_transcript_columns(input_file, output_file, columns_to_expand,sep1,sep2):
    """
    Expands multiple HGVS-style columns into a wide format.
    columns_to_expand: List of columns, e.g., ['HGVSc', 'HGVSp']
    """
    df = pd.read_csv(input_file,sep="\t")

    # This will hold all the newly created DataFrames for merging at the end
    expanded_dataframes = [df]

    for col in columns_to_expand:
        if col not in df.columns:
            print(f"Warning: Column '{col}' not found. Skipping.")
            continue

        print(f"Parsing {col}...")

        def parse_hgvs_string(hgvs_str):
            if pd.isna(hgvs_str) or hgvs_str == "":
                return {}
            
            entries = str(hgvs_str).split(sep2)
            row_data = {}
            for entry in entries:
                if sep1 in entry:
                    # transcript = 'NM_000277.3', change = 'c.1155C>G'
                    transcript, change = entry.split(sep1, 1)
                    # We prefix the column name to keep HGVSc separate from HGVSp
                    row_data[f"{col}|{transcript}"] = change
            return row_data

        # 1. Create a list of dictionaries for this specific column
        dicts = df[col].apply(parse_hgvs_string).tolist()
        
        # 2. Convert to a DataFrame
        new_df = pd.DataFrame(dicts)
        
        # 3. Store it for concatenation
        expanded_dataframes.append(new_df)

    # Combine the original DF with all new transcript-specific columns
    final_df = pd.concat(expanded_dataframes, axis=1)

    # drop original columns
    final_df = final_df.drop(columns=columns_to_expand)

    # Save results
    final_df.to_csv(output_file, index=False,sep="\t")
    
    new_col_count = len(final_df.columns) - len(df.columns)
    print(f"Done! Added {new_col_count} new transcript-specific columns.")

if __name__ == "__main__":
    # Specify the columns you want to burst open
    target_columns = ['INFO']
    
    expand_transcript_columns(
        'vr_output1.vcf', 
        'vr_output2.vcf', 
        target_columns,
        sep1='=',
        sep2=';'
    )
    
if __name__ == "__main__":
    # Specify the columns you want to burst open
    target_columns = ['INFO|HGVSg','INFO|HGVSc','INFO|HGVSp','INFO|SPDI','INFO|Variant_synonyms']
    
    expand_transcript_columns(
        'vr_output2.vcf', 
        'vr_output3.vcf', 
        target_columns,
        sep1=':',
        sep2=','
    )

Parsing INFO...
Done! Added 6 new transcript-specific columns.
Parsing INFO|HGVSg...
Parsing INFO|HGVSc...
Parsing INFO|HGVSp...
Parsing INFO|SPDI...
Parsing INFO|Variant_synonyms...
Done! Added 52 new transcript-specific columns.


In [2]:
# extracting only rsIDs from existing variation column
import pandas as pd
import re
df_variants = pd.read_csv("variants_color_coded.csv")
def extract_rsid(text):
    if pd.isna(text) or text == "":
        return ""
    match = re.search(r'\brs\d+\b', str(text))
    return match.group(0) if match else ""
source_col = 'Existing variation'
if source_col in df_variants.columns:
    df_variants['ID'] = df_variants[source_col].apply(extract_rsid)
else:
    print(f"Column {source_col} not found!")

pd.set_option('display.max_columns', None)

df_anno = pd.read_csv("vr_output3.vcf",sep="\t")
df_anno['#CHROM'] = df_anno['#CHROM'].replace(12, 'chr12')
df_anno = df_anno.rename(columns={'#CHROM': 'Chromosome', 'POS': 'Position','REF':'Ref','ALT':'Alt'})

# merging by rsID (too many variants, some unncesary)
indel_merge = pd.merge(df_variants[df_variants["Variant type"] != "SNV"], df_anno, on=['ID'], how='left')

# merging by position + ref and alt; removes indels
snv_merge = pd.merge(df_variants[df_variants["Variant type"] == "SNV"], df_anno, on=['Chromosome','Position','Ref','Alt'], how='left')

#print(merged_df)
#merged_df.to_csv("with_anno.csv")
indel_merge

# need to fix the problem where i have multiple ref/alt alleles if i merge by rsID but if i merge by specific ref alt, my indels get messed up (some are not included)
# might want to still merge by positions but make exception for indels? or fix indels manually?

,Sample,Amplicon,Chromosome_x,Position_x,Ref_x,Alt_x,ID 1,ID 2,ID 3,ID 4,ID 5,ID 6,Variant type,Gene,Gene symbol,Feature,Feature type,Exon,Intron,Consequence,Impact,Existing variation,Clinical significance (ClinVar),PubMed,SIFT,PolyPhen,Codons,cDNA effect,Protein effect,1000 Genomes frequency,gnomAD frequency,Offset from primer end,5' context,Alleles,3' context,Indel length,ID,Unnamed: 0,Chromosome_y,Position_y,Ref_y,Alt_y,QUAL,FILTER,INFO|VARID,INFO|VCF,INFO|HGVSg|NC_000012.12,INFO|HGVSc|ENST00000307000.7,INFO|HGVSc|ENST00000549247.6,INFO|HGVSc|ENST00000551114.2,INFO|HGVSc|ENST00000553106.6,INFO|HGVSc|ENST00000635477.1,INFO|HGVSc|ENST00000635528.1,INFO|HGVSc|ENST00000906692.1,INFO|HGVSc|ENST00000906693.1,INFO|HGVSc|ENST00000906694.1,INFO|HGVSc|ENST00000906695.1,INFO|HGVSc|ENST00000906696.1,INFO|HGVSc|ENST00000906697.1,INFO|HGVSc|NM_000277.3,INFO|HGVSc|NM_001354304.2,INFO|HGVSc|ENST00000549111.5,INFO|HGVSc|XM_017019370.2,INFO|HGVSc|ENST00000551988.5,INFO|HGVSc|ENST00000550978.6,INFO|HGVSc|XR_007063428.1,INFO|HGVSc|ENST00000551337.5,INFO|HGVSc|ENST00000546844.1,INFO|HGVSc|ENST00000548677.2,INFO|HGVSc|ENST00000548928.1,INFO|HGVSc|ENST00000635500.1,INFO|HGVSc|ENST00000546708.5,INFO|HGVSc|ENST00000547319.1,INFO|HGVSp|ENSP00000303500.2,INFO|HGVSp|ENSP00000448059.1,INFO|HGVSp|ENSP00000489230.1,INFO|HGVSp|ENSP00000576751.1,INFO|HGVSp|ENSP00000576752.1,INFO|HGVSp|ENSP00000576753.1,INFO|HGVSp|ENSP00000576754.1,INFO|HGVSp|ENSP00000576755.1,INFO|HGVSp|ENSP00000576756.1,INFO|HGVSp|NP_000268.1,INFO|HGVSp|NP_001341233.1,INFO|HGVSp|XP_016874859.1,INFO|HGVSp|ENSP00000446658.1,INFO|HGVSp|ENSP00000489016.1,INFO|HGVSp|ENSP00000447620.1,INFO|SPDI|NC_000012.12,INFO|Variant_synonyms|ClinVar,INFO|Variant_synonyms|PAHdb,INFO|Variant_synonyms|ArchivedbSNP,INFO|Variant_synonyms|PhenCode,INFO|Variant_synonyms|dbSNPHGVS,INFO|Variant_synonyms|NM_001127179.2,INFO|Variant_synonyms|NM_000260.3,INFO|Variant_synonyms|NM_001127180.2,INFO|Variant_synonyms|NM_000260.4,INFO|Variant_synonyms|NM_001127180.1,INFO|Variant_synonyms|NM_001369365.1,INFO|Variant_synonyms|UniProt,INFO|Variant_synonyms|OMIM,INFO|Variant_synonyms|PharmGKB
0,PAH-14_S4_,pool2-10,chr12,102894876,GAGA,G,PAH-14_S4_,NaN,NaN,NaN,NaN,NaN,deletion,ENSG00000171759,PAH,ENST00000553106,Transcript,3 of 13,NaN,inframe_deletion,moderate,rs62642094,"not_provided, pathogenic","15503242, 24401910, 16256386, 23932990, 986030...",NaN,NaN,TCT/-,NaN,NaN,NaN,0.000007,102,TAAAC,GAGA/G,AGGTC,-3.0,rs62642094,21,chr12,102894878,GAAG,G,.,.,rs62642094,12-102894878-GAAG-G,g.102894879_102894881del,c.191_193del,NaN,NaN,c.206_208del,NaN,NaN,c.206_208del,c.206_208del,c.206_208del,c.206_208del,c.98_100del,c.168+17910_168+17912del,c.206_208del,c.206_208del,n.302_304del,c.206_208del,n.295_297del,c.190_192del,n.863-9819_863-9817del,c.206_208del,c.206_208del,n.293_295del,n.128_130del,n.174_176del,NaN,NaN,p.Ser65del,p.Ser70del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser34del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser65del,p.Ser70del,102894878:AAG:,RCV000088875,p.S70del,rs199475620,PAH_c.206_208delCTT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PAH-15_S5_,pool2-10,chr12,102894876,GAGA,G,PAH-15_S5_,NaN,NaN,NaN,NaN,NaN,deletion,ENSG00000171759,PAH,ENST00000553106,Transcript,3 of 13,NaN,inframe_deletion,moderate,rs62642094,"not_provided, pathogenic","15503242, 24401910, 16256386, 23932990, 986030...",NaN,NaN,TCT/-,NaN,NaN,NaN,0.000007,102,TAAAC,GAGA/G,AGGTC,-3.0,rs62642094,21,chr12,102894878,GAAG,G,.,.,rs62642094,12-102894878-GAAG-G,g.102894879_102894881del,c.191_193del,NaN,NaN,c.206_208del,NaN,NaN,c.206_208del,c.206_208del,c.206_208del,c.206_208del,c.98_100del,c.168+17910_168+17912del,c.206_208del,c.206_208del,n.302_304del,c.206_208del,n.295_297del,c.190_192del,n.863-9819_863-9817del,c.206_208del,c.206_208del,n.293_295del,n.128_130del,n.174_176del,NaN,NaN,p.Ser65del,p.Ser70del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser34del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser65del,p.Ser70del,102894878:A

In [3]:
df_variants[["Position", "Ref", "Alt", "ID"]].drop_duplicates().values

array([[102844478, 'T', 'G', ''],
       [102877572, 'G', 'A', 'rs2037639'],
       [102894811, 'T', 'G', ''],
       [102843690, 'G', 'C', 'rs772897'],
       [102917201, 'T', 'G', 'rs2280615'],
       [102912801, 'C', 'T', 'rs118092776'],
       [102840329, 'G', 'T', ''],
       [102912772, 'A', 'G', 'rs17842947'],
       [102855146, 'C', 'T', 'rs1126758'],
       [102852922, 'C', 'T', 'rs1042503'],
       [102852929, 'C', 'T', 'rs62508588'],
       [102846852, 'C', 'A', 'rs1522306'],
       [102855386, 'C', 'T', 'rs2251905'],
       [102866641, 'C', 'T', 'rs199475663'],
       [102877415, 'G', 'A', 'rs1718301'],
       [102894812, 'G', 'A', 'rs62514903'],
       [102840399, 'C', 'T', 'rs5030861'],
       [102840507, 'G', 'A', 'rs5030857'],
       [102894876, 'GAGA', 'G', 'rs62642094'],
       [102843686, 'A', 'G', 'rs62517194'],
       [102843790, 'C', 'T', 'rs5030855'],
       [102838988, 'C', 'T', 'rs1801153'],
       [102917584, 'GC', 'G', 'rs113191080'],
       [102917606, 'ACAG

In [4]:
# Filter rows where ID_x and ID_y differ, then select only those two columns
snv_merge[snv_merge["ID_x"] != snv_merge["ID_y"]][["ID_x", "ID_y"]]
snv_merge2 = snv_merge.drop(columns=['ID_x'])
snv_merge3 = snv_merge2.rename(columns={'ID_y': 'ID'})
snv_merge3.columns

Index(['Sample', 'Amplicon', 'Chromosome', 'Position', 'Ref', 'Alt', 'ID 1',
       'ID 2', 'ID 3', 'ID 4', 'ID 5', 'ID 6', 'Variant type', 'Gene',
       'Gene symbol', 'Feature', 'Feature type', 'Exon', 'Intron',
       'Consequence', 'Impact', 'Existing variation',
       'Clinical significance (ClinVar)', 'PubMed', 'SIFT', 'PolyPhen',
       'Codons', 'cDNA effect', 'Protein effect', '1000 Genomes frequency',
       'gnomAD frequency', 'Offset from primer end', '5' context', 'Alleles',
       '3' context', 'Indel length', 'Unnamed: 0', 'ID', 'QUAL', 'FILTER',
       'INFO|VARID', 'INFO|VCF', 'INFO|HGVSg|NC_000012.12',
       'INFO|HGVSc|ENST00000307000.7', 'INFO|HGVSc|ENST00000549247.6',
       'INFO|HGVSc|ENST00000551114.2', 'INFO|HGVSc|ENST00000553106.6',
       'INFO|HGVSc|ENST00000635477.1', 'INFO|HGVSc|ENST00000635528.1',
       'INFO|HGVSc|ENST00000906692.1', 'INFO|HGVSc|ENST00000906693.1',
       'INFO|HGVSc|ENST00000906694.1', 'INFO|HGVSc|ENST00000906695.1',
       'INFO|HG

In [5]:
# shows which columns have _x and _y in them
print([col for col in indel_merge.columns if '_x' in col or '_y' in col])

#indel_merge[indel_merge["Chromosome_x"] != indel_merge["Chromosome_y"]][["Chromosome_x", "Chromosome_y"]]
# no difference in chromosome
print(indel_merge[indel_merge["Position_x"] != indel_merge["Position_y"]][["Position_x", "Position_y","Variant type"]])
print(indel_merge[indel_merge["Ref_x"] != indel_merge["Ref_y"]][["Ref_x", "Ref_y","Variant type"]])
print(indel_merge[indel_merge["Alt_x"] != indel_merge["Alt_y"]][["Alt_x", "Alt_y","Variant type"]])

# the mismatch in indels is only in ref alt and position. i will use position, ref and alt from VEP since they are left normalized
# also! need to figure out why annotation had 5 unique deletions and merge only included 4 - answer - it's because one of mu mutations can be deletion or insertion and both have the same rsID

# i will use y data (from annotation) and remove from annotation 
indel_merge2 = indel_merge
indel_merge2 = indel_merge.drop(columns=['Position_x',"Ref_x", "Chromosome_x", "Alt_x"])
indel_merge3 = indel_merge2.rename(columns={"Chromosome_y":"Chromosome","Position_y":"Position","Ref_y":"Ref","Alt_y":"Alt"})
indel_merge3

['Chromosome_x', 'Position_x', 'Ref_x', 'Alt_x', 'Chromosome_y', 'Position_y', 'Ref_y', 'Alt_y']
   Position_x  Position_y Variant type
0   102894876   102894878     deletion
1   102894876   102894878     deletion
2   102917584   102917585     deletion
3   102917606   102917607     deletion
4   102855176   102855177     deletion
   Ref_x  Ref_y Variant type
0   GAGA   GAAG     deletion
1   GAGA   GAAG     deletion
2     GC     CC     deletion
3  ACAGT  CAGTC     deletion
4    ATC    TCT     deletion
  Alt_x Alt_y Variant type
2     G     C     deletion
3     A     C     deletion
4     A     T     deletion


,Sample,Amplicon,ID 1,ID 2,ID 3,ID 4,ID 5,ID 6,Variant type,Gene,Gene symbol,Feature,Feature type,Exon,Intron,Consequence,Impact,Existing variation,Clinical significance (ClinVar),PubMed,SIFT,PolyPhen,Codons,cDNA effect,Protein effect,1000 Genomes frequency,gnomAD frequency,Offset from primer end,5' context,Alleles,3' context,Indel length,ID,Unnamed: 0,Chromosome,Position,Ref,Alt,QUAL,FILTER,INFO|VARID,INFO|VCF,INFO|HGVSg|NC_000012.12,INFO|HGVSc|ENST00000307000.7,INFO|HGVSc|ENST00000549247.6,INFO|HGVSc|ENST00000551114.2,INFO|HGVSc|ENST00000553106.6,INFO|HGVSc|ENST00000635477.1,INFO|HGVSc|ENST00000635528.1,INFO|HGVSc|ENST00000906692.1,INFO|HGVSc|ENST00000906693.1,INFO|HGVSc|ENST00000906694.1,INFO|HGVSc|ENST00000906695.1,INFO|HGVSc|ENST00000906696.1,INFO|HGVSc|ENST00000906697.1,INFO|HGVSc|NM_000277.3,INFO|HGVSc|NM_001354304.2,INFO|HGVSc|ENST00000549111.5,INFO|HGVSc|XM_017019370.2,INFO|HGVSc|ENST00000551988.5,INFO|HGVSc|ENST00000550978.6,INFO|HGVSc|XR_007063428.1,INFO|HGVSc|ENST00000551337.5,INFO|HGVSc|ENST00000546844.1,INFO|HGVSc|ENST00000548677.2,INFO|HGVSc|ENST00000548928.1,INFO|HGVSc|ENST00000635500.1,INFO|HGVSc|ENST00000546708.5,INFO|HGVSc|ENST00000547319.1,INFO|HGVSp|ENSP00000303500.2,INFO|HGVSp|ENSP00000448059.1,INFO|HGVSp|ENSP00000489230.1,INFO|HGVSp|ENSP00000576751.1,INFO|HGVSp|ENSP00000576752.1,INFO|HGVSp|ENSP00000576753.1,INFO|HGVSp|ENSP00000576754.1,INFO|HGVSp|ENSP00000576755.1,INFO|HGVSp|ENSP00000576756.1,INFO|HGVSp|NP_000268.1,INFO|HGVSp|NP_001341233.1,INFO|HGVSp|XP_016874859.1,INFO|HGVSp|ENSP00000446658.1,INFO|HGVSp|ENSP00000489016.1,INFO|HGVSp|ENSP00000447620.1,INFO|SPDI|NC_000012.12,INFO|Variant_synonyms|ClinVar,INFO|Variant_synonyms|PAHdb,INFO|Variant_synonyms|ArchivedbSNP,INFO|Variant_synonyms|PhenCode,INFO|Variant_synonyms|dbSNPHGVS,INFO|Variant_synonyms|NM_001127179.2,INFO|Variant_synonyms|NM_000260.3,INFO|Variant_synonyms|NM_001127180.2,INFO|Variant_synonyms|NM_000260.4,INFO|Variant_synonyms|NM_001127180.1,INFO|Variant_synonyms|NM_001369365.1,INFO|Variant_synonyms|UniProt,INFO|Variant_synonyms|OMIM,INFO|Variant_synonyms|PharmGKB
0,PAH-14_S4_,pool2-10,PAH-14_S4_,NaN,NaN,NaN,NaN,NaN,deletion,ENSG00000171759,PAH,ENST00000553106,Transcript,3 of 13,NaN,inframe_deletion,moderate,rs62642094,"not_provided, pathogenic","15503242, 24401910, 16256386, 23932990, 986030...",NaN,NaN,TCT/-,NaN,NaN,NaN,0.000007,102,TAAAC,GAGA/G,AGGTC,-3.0,rs62642094,21,chr12,102894878,GAAG,G,.,.,rs62642094,12-102894878-GAAG-G,g.102894879_102894881del,c.191_193del,NaN,NaN,c.206_208del,NaN,NaN,c.206_208del,c.206_208del,c.206_208del,c.206_208del,c.98_100del,c.168+17910_168+17912del,c.206_208del,c.206_208del,n.302_304del,c.206_208del,n.295_297del,c.190_192del,n.863-9819_863-9817del,c.206_208del,c.206_208del,n.293_295del,n.128_130del,n.174_176del,NaN,NaN,p.Ser65del,p.Ser70del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser34del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser65del,p.Ser70del,102894878:AAG:,RCV000088875,p.S70del,rs199475620,PAH_c.206_208delCTT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PAH-15_S5_,pool2-10,PAH-15_S5_,NaN,NaN,NaN,NaN,NaN,deletion,ENSG00000171759,PAH,ENST00000553106,Transcript,3 of 13,NaN,inframe_deletion,moderate,rs62642094,"not_provided, pathogenic","15503242, 24401910, 16256386, 23932990, 986030...",NaN,NaN,TCT/-,NaN,NaN,NaN,0.000007,102,TAAAC,GAGA/G,AGGTC,-3.0,rs62642094,21,chr12,102894878,GAAG,G,.,.,rs62642094,12-102894878-GAAG-G,g.102894879_102894881del,c.191_193del,NaN,NaN,c.206_208del,NaN,NaN,c.206_208del,c.206_208del,c.206_208del,c.206_208del,c.98_100del,c.168+17910_168+17912del,c.206_208del,c.206_208del,n.302_304del,c.206_208del,n.295_297del,c.190_192del,n.863-9819_863-9817del,c.206_208del,c.206_208del,n.293_295del,n.128_130del,n.174_176del,NaN,NaN,p.Ser65del,p.Ser70del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser34del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser65del,p.Ser70del,102894878:AAG:,RCV000088875,p.S70del,rs199475620,PAH_c.206_208delCTT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [6]:
# Create a comparison dataframe using an outer merge on all keys
comparison = pd.merge(
    df_variants, 
    df_anno, 
    on=['Chromosome', 'Position', 'Ref', 'Alt', 'ID'], 
    how='outer', 
    indicator=True
)

# Rows in df_variants (left) that didn't find a match in df_anno (right)
missing_in_anno = comparison[comparison['_merge'] == 'left_only']
print(missing_in_anno[missing_in_anno['Variant type'] =='SNV'].shape[0])
print(missing_in_anno[missing_in_anno['Variant type'] =='deletion'].shape[0])
print(missing_in_anno.shape)
missing_in_anno[missing_in_anno["Variant type"] != "SNV"]

39
5
(44, 100)


,Sample,Amplicon,Chromosome,Position,Ref,Alt,ID 1,ID 2,ID 3,ID 4,ID 5,ID 6,Variant type,Gene,Gene symbol,Feature,Feature type,Exon,Intron,Consequence,Impact,Existing variation,Clinical significance (ClinVar),PubMed,SIFT,PolyPhen,Codons,cDNA effect,Protein effect,1000 Genomes frequency,gnomAD frequency,Offset from primer end,5' context,Alleles,3' context,Indel length,ID,Unnamed: 0,QUAL,FILTER,INFO|VARID,INFO|VCF,INFO|HGVSg|NC_000012.12,INFO|HGVSc|ENST00000307000.7,INFO|HGVSc|ENST00000549247.6,INFO|HGVSc|ENST00000551114.2,INFO|HGVSc|ENST00000553106.6,INFO|HGVSc|ENST00000635477.1,INFO|HGVSc|ENST00000635528.1,INFO|HGVSc|ENST00000906692.1,INFO|HGVSc|ENST00000906693.1,INFO|HGVSc|ENST00000906694.1,INFO|HGVSc|ENST00000906695.1,INFO|HGVSc|ENST00000906696.1,INFO|HGVSc|ENST00000906697.1,INFO|HGVSc|NM_000277.3,INFO|HGVSc|NM_001354304.2,INFO|HGVSc|ENST00000549111.5,INFO|HGVSc|XM_017019370.2,INFO|HGVSc|ENST00000551988.5,INFO|HGVSc|ENST00000550978.6,INFO|HGVSc|XR_007063428.1,INFO|HGVSc|ENST00000551337.5,INFO|HGVSc|ENST00000546844.1,INFO|HGVSc|ENST00000548677.2,INFO|HGVSc|ENST00000548928.1,INFO|HGVSc|ENST00000635500.1,INFO|HGVSc|ENST00000546708.5,INFO|HGVSc|ENST00000547319.1,INFO|HGVSp|ENSP00000303500.2,INFO|HGVSp|ENSP00000448059.1,INFO|HGVSp|ENSP00000489230.1,INFO|HGVSp|ENSP00000576751.1,INFO|HGVSp|ENSP00000576752.1,INFO|HGVSp|ENSP00000576753.1,INFO|HGVSp|ENSP00000576754.1,INFO|HGVSp|ENSP00000576755.1,INFO|HGVSp|ENSP00000576756.1,INFO|HGVSp|NP_000268.1,INFO|HGVSp|NP_001341233.1,INFO|HGVSp|XP_016874859.1,INFO|HGVSp|ENSP00000446658.1,INFO|HGVSp|ENSP00000489016.1,INFO|HGVSp|ENSP00000447620.1,INFO|SPDI|NC_000012.12,INFO|Variant_synonyms|ClinVar,INFO|Variant_synonyms|PAHdb,INFO|Variant_synonyms|ArchivedbSNP,INFO|Variant_synonyms|PhenCode,INFO|Variant_synonyms|dbSNPHGVS,INFO|Variant_synonyms|NM_001127179.2,INFO|Variant_synonyms|NM_000260.3,INFO|Variant_synonyms|NM_001127180.2,INFO|Variant_synonyms|NM_000260.4,INFO|Variant_synonyms|NM_001127180.1,INFO|Variant_synonyms|NM_001369365.1,INFO|Variant_synonyms|UniProt,INFO|Variant_synonyms|OMIM,INFO|Variant_synonyms|PharmGKB,_merge
212,PAH-31_S21_,pool1-8,chr12,102855176,ATC,A,PAH-31_S21_,NaN,NaN,NaN,NaN,NaN,deletion,ENSG00000171759,PAH,ENST00000553106,Transcript,6 of 13,NaN,frameshift,high,rs62514936,pathogenic,"10394930, 24350308, 17935162, 23074961, 233575...",NaN,NaN,GAt/t,NaN,NaN,NaN,0.000013,80.0,ATGTT,ATC/A,TTCAT,-2.0,rs62514936,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
358,PAH-14_S4_,pool2-10,chr12,102894876,GAGA,G,PAH-14_S4_,NaN,NaN,NaN,NaN,NaN,deletion,ENSG00000171759,PAH,ENST00000553106,Transcript,3 of 13,NaN,inframe_deletion,moderate,rs62642094,"not_provided, pathogenic","15503242, 24401910, 16256386, 23932990, 986030...",NaN,NaN,TCT/-,NaN,NaN,NaN,0.000007,102.0,TAAAC,GAGA/G,AGGTC,-3.0,rs62642094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
359,PAH-15_S5_,pool2-10,chr12,102894876,GAGA,G,PAH-15_S5_,NaN,NaN,NaN,NaN,NaN,deletion,ENSG00000171759,PAH,ENST00000553106,Transcript,3 of 13,NaN,inframe_deletion,moderate,rs62642094,"not_provided, pathogenic","15503242, 24401910, 16256386, 23932990, 986030...",NaN,NaN,TCT/-,NaN,NaN,NaN,0.000007,102.0,TAAAC,GAGA/G,AGGTC,-3.0,rs62642094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
389,PAH-20_S10_,pool2-13,chr12,102917584,GC,G,PAH-20_S10_,NaN,NaN,NaN,NaN,NaN,deletion,ENSG00000171759,PAH,ENST00000553106,Transcript,NaN,NaN,upstrea

In [7]:
# combining indel merge (by rsID) and snv merge (by position + rsID)
final_df = pd.concat([indel_merge3, snv_merge3], ignore_index=True)
final_df

,Sample,Amplicon,ID 1,ID 2,ID 3,ID 4,ID 5,ID 6,Variant type,Gene,Gene symbol,Feature,Feature type,Exon,Intron,Consequence,Impact,Existing variation,Clinical significance (ClinVar),PubMed,SIFT,PolyPhen,Codons,cDNA effect,Protein effect,1000 Genomes frequency,gnomAD frequency,Offset from primer end,5' context,Alleles,3' context,Indel length,ID,Unnamed: 0,Chromosome,Position,Ref,Alt,QUAL,FILTER,INFO|VARID,INFO|VCF,INFO|HGVSg|NC_000012.12,INFO|HGVSc|ENST00000307000.7,INFO|HGVSc|ENST00000549247.6,INFO|HGVSc|ENST00000551114.2,INFO|HGVSc|ENST00000553106.6,INFO|HGVSc|ENST00000635477.1,INFO|HGVSc|ENST00000635528.1,INFO|HGVSc|ENST00000906692.1,INFO|HGVSc|ENST00000906693.1,INFO|HGVSc|ENST00000906694.1,INFO|HGVSc|ENST00000906695.1,INFO|HGVSc|ENST00000906696.1,INFO|HGVSc|ENST00000906697.1,INFO|HGVSc|NM_000277.3,INFO|HGVSc|NM_001354304.2,INFO|HGVSc|ENST00000549111.5,INFO|HGVSc|XM_017019370.2,INFO|HGVSc|ENST00000551988.5,INFO|HGVSc|ENST00000550978.6,INFO|HGVSc|XR_007063428.1,INFO|HGVSc|ENST00000551337.5,INFO|HGVSc|ENST00000546844.1,INFO|HGVSc|ENST00000548677.2,INFO|HGVSc|ENST00000548928.1,INFO|HGVSc|ENST00000635500.1,INFO|HGVSc|ENST00000546708.5,INFO|HGVSc|ENST00000547319.1,INFO|HGVSp|ENSP00000303500.2,INFO|HGVSp|ENSP00000448059.1,INFO|HGVSp|ENSP00000489230.1,INFO|HGVSp|ENSP00000576751.1,INFO|HGVSp|ENSP00000576752.1,INFO|HGVSp|ENSP00000576753.1,INFO|HGVSp|ENSP00000576754.1,INFO|HGVSp|ENSP00000576755.1,INFO|HGVSp|ENSP00000576756.1,INFO|HGVSp|NP_000268.1,INFO|HGVSp|NP_001341233.1,INFO|HGVSp|XP_016874859.1,INFO|HGVSp|ENSP00000446658.1,INFO|HGVSp|ENSP00000489016.1,INFO|HGVSp|ENSP00000447620.1,INFO|SPDI|NC_000012.12,INFO|Variant_synonyms|ClinVar,INFO|Variant_synonyms|PAHdb,INFO|Variant_synonyms|ArchivedbSNP,INFO|Variant_synonyms|PhenCode,INFO|Variant_synonyms|dbSNPHGVS,INFO|Variant_synonyms|NM_001127179.2,INFO|Variant_synonyms|NM_000260.3,INFO|Variant_synonyms|NM_001127180.2,INFO|Variant_synonyms|NM_000260.4,INFO|Variant_synonyms|NM_001127180.1,INFO|Variant_synonyms|NM_001369365.1,INFO|Variant_synonyms|UniProt,INFO|Variant_synonyms|OMIM,INFO|Variant_synonyms|PharmGKB
0,PAH-14_S4_,pool2-10,PAH-14_S4_,NaN,NaN,NaN,NaN,NaN,deletion,ENSG00000171759,PAH,ENST00000553106,Transcript,3 of 13,NaN,inframe_deletion,moderate,rs62642094,"not_provided, pathogenic","15503242, 24401910, 16256386, 23932990, 986030...",NaN,NaN,TCT/-,NaN,NaN,NaN,0.000007,102,TAAAC,GAGA/G,AGGTC,-3.0,rs62642094,21.0,chr12,102894878,GAAG,G,.,.,rs62642094,12-102894878-GAAG-G,g.102894879_102894881del,c.191_193del,NaN,NaN,c.206_208del,NaN,NaN,c.206_208del,c.206_208del,c.206_208del,c.206_208del,c.98_100del,c.168+17910_168+17912del,c.206_208del,c.206_208del,n.302_304del,c.206_208del,n.295_297del,c.190_192del,n.863-9819_863-9817del,c.206_208del,c.206_208del,n.293_295del,n.128_130del,n.174_176del,NaN,NaN,p.Ser65del,p.Ser70del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser34del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser65del,p.Ser70del,102894878:AAG:,RCV000088875,p.S70del,rs199475620,PAH_c.206_208delCTT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PAH-15_S5_,pool2-10,PAH-15_S5_,NaN,NaN,NaN,NaN,NaN,deletion,ENSG00000171759,PAH,ENST00000553106,Transcript,3 of 13,NaN,inframe_deletion,moderate,rs62642094,"not_provided, pathogenic","15503242, 24401910, 16256386, 23932990, 986030...",NaN,NaN,TCT/-,NaN,NaN,NaN,0.000007,102,TAAAC,GAGA/G,AGGTC,-3.0,rs62642094,21.0,chr12,102894878,GAAG,G,.,.,rs62642094,12-102894878-GAAG-G,g.102894879_102894881del,c.191_193del,NaN,NaN,c.206_208del,NaN,NaN,c.206_208del,c.206_208del,c.206_208del,c.206_208del,c.98_100del,c.168+17910_168+17912del,c.206_208del,c.206_208del,n.302_304del,c.206_208del,n.295_297del,c.190_192del,n.863-9819_863-9817del,c.206_208del,c.206_208del,n.293_295del,n.128_130del,n.174_176del,NaN,NaN,p.Ser65del,p.Ser70del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser34del,NaN,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser70del,p.Ser65del,p.Ser70del,102894878:AAG:,RCV000088875,p.S70del,rs199475620,PAH_c.206_208delCTT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [8]:
final_df.to_csv("output_w_annotation.tsv", index=False,sep="\t")